__Definition__<br>
Hallucination = text that looks fluent but is not grounded by evidence. 

__Hallucination types__
- Input based (faithfulness)
    - intrinsic = incorrect processing of the data from source
    - extrinsic = totaly made up
- Factual (truthfulness)<br>
- Logical<br>wrong reasoning
- Reference<br>made up links to sources
- Sycophancy<br>repeating incorrect conclusion by the user
- Calibration failures<br>failure to admit uncertainty: "Are you sure? Yes, absolutely"
- RAG-specific<br>
- Translation based

Possible Triggers:
- Objective mismatch <br>misalginment, bias towards fluency over factuality
- Poor pretraining data
- Long context effects

Evaluation
- TruthfulQA<br>labeled dataset with 800 questions containing common misconceptions
- FRANK
- FactCC (2020)<br>[[paper]](https://aclanthology.org/2020.emnlp-main.750.pdf?utm_source=chatgpt.com) classifier by Salesforce trained to detect inconsistencies in a (Q,D) pair
- HaluEval<br>labeled dataset with 35K questions with generations labeled with hallucinations
- FactScore (2023)<br>[[paper]](https://arxiv.org/pdf/2305.14251) A metric that shows the ratio of claims that are grounded/approved by LLM-as-a-judge or human labeler
- RAGAS
- SelfCheckGPT (2023)<br>[[paper]](https://arxiv.org/abs/2303.08896?utm_source=chatgpt.com)<br>method of asking the model to generate K completions and evaluate its consistency

Claim = statement that can be evalauted. Generation can consist of 0, 1 or many claims

Strategies for fighting hallucainations:
1) __Use RAG__ - it will ground claims if they can be extracted from the source<br>proper configuration necessary, otherwise will fail
2) __CoVE (self-verification)__ - the model before finalizing generation
3) __Post-editing__<br>run a separate critic model after generation that would evaluate consistency of the answer
4) __Uncertainty calibration__<br>make model admit its uncertainty by defining bahaviour in the prompt



Chain-of-Verifications (2023)
---
[[paper]](https://arxiv.org/pdf/2309.11495)

CoVE = Chain-of-Verifications<Br>

<img src="img/halu_cove.png" width=500>


# SelfRAG (2023)
---
[[paper]](https://arxiv.org/pdf/2310.11511)

__Idea:__ let the model decide whether it needs to apply RAG to fetch info during generation

We generate iteratively - sentence-by-sentence we append new continuations to final answer. Before each step we run a classifer that predicts whether knowledge retrieval is required. If "yes" it fetches new documents and ranks them according to relevance, consistency and usefulness and store top N documents in memory (including previously fetched). 

Next the model generates k candidate completions. If answer is useful it stops generation

<img src="img/halu_self_rag.png" width=750>

Algorithm:

0) After finishing the generation check the logit of RET token. If RET=YES, run retrieval now, merge/rerank evidence E
1) Generate the next sentence unitl you hit a sentence boundary
2) Ask the model for the reflection tokens about that just-drafted sentence:
ISREL: for each top passage in E, how relevant is it to this sentence?
ISSUP: does the passage support / partially support / not support / contradict the sentence?
ISUSE: how useful is this sentence for answering the question?
Aggregate those signals (e.g., max support over passages, or relevance-weighted).
Score the candidate sentence = (fluency/logprob) + (support) + (usefulness).
If you’re in strict mode, reject sentences that aren’t sufficiently supported and (optionally) force another retrieval.
4) Append the best-scored, sufficiently supported sentence to the answer
5) Repeat

Go back to step 0 for the next sentence until you emit an end-of-answer token or hit a length cap.